# Capstone Project — Bayesian Black-Box Optimisation (BBO)
## Imperial College PCMLAI · Stage 2 · All 8 Functions

This notebook implements a **systematic Bayesian Optimisation pipeline** for all eight black-box functions in the BBO Capstone project.

**Strategy:**
- Weeks 1–4: UCB with high β (exploration — map the unknown space)
- Weeks 5–9: EI (balanced — exploit promising regions intelligently)
- Weeks 10–13: EI / UCB with low β (exploitation — refine the best found)

**Workflow per week:**
1. Load current `.npy` data files
2. Fit Gaussian Process surrogate model
3. Compute acquisition function → get next query point
4. Submit to portal → receive y output via email
5. Append new (x, y) to data files → save
6. Write weekly reflection


In [ ]:
# ── Core imports ──────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
from pathlib import Path

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel as C
from scipy.stats import norm
from scipy.optimize import differential_evolution

warnings.filterwarnings('ignore')
np.random.seed(42)

print("✓ All imports successful")
print(f"  numpy  {np.__version__}")


## 1. Configuration — Data Paths and Function Registry

In [ ]:
# ── DATA PATHS — edit this to match your folder structure ────────────────────
# Place your .npy files like:
#   data/function_1/initial_inputs.npy
#   data/function_1/initial_outputs.npy
#   data/function_2/initial_inputs.npy  ... etc.

DATA_ROOT = Path("data")   # <── change if your folder is different

# ── FUNCTION REGISTRY ─────────────────────────────────────────────────────────
FUNCTIONS = {
    1: {
        "dim": 2, "n_init": 10,
        "description": "Radiation source detection in a 2D area",
        "notes": "Sharp, localised peak. All-zero initial outputs typical. Pure exploration first."
    },
    2: {
        "dim": 2, "n_init": 10,
        "description": "Noisy ML model log-likelihood (2D)",
        "notes": "Noisy outputs — EI handles noise better than UCB. Multiple local optima possible."
    },
    3: {
        "dim": 3, "n_init": 15,
        "description": "Drug discovery — minimise adverse reactions (3D)",
        "notes": "Negated for maximisation. Negative outputs expected."
    },
    4: {
        "dim": 4, "n_init": 30,
        "description": "Warehouse placement optimisation (4D)",
        "notes": "Dynamic, many local optima. Use high exploration early."
    },
    5: {
        "dim": 4, "n_init": 20,
        "description": "Chemical yield maximisation — unimodal (4D)",
        "notes": "Single peak. Once region found, switch to exploitation quickly."
    },
    6: {
        "dim": 5, "n_init": 20,
        "description": "Cake recipe optimisation — negative score by design (5D)",
        "notes": "Output is negative by design. Maximise (bring closest to zero)."
    },
    7: {
        "dim": 6, "n_init": 30,
        "description": "ML hyperparameter tuning — 6 parameters (6D)",
        "notes": "High-dimensional. Consider literature on good HP ranges."
    },
    8: {
        "dim": 8, "n_init": 40,
        "description": "8-parameter ML model optimisation (8D)",
        "notes": "High-dimensional, complex landscape. Global optimum hard — find strong local maxima."
    },
}

# ── GP HYPERPARAMETERS ────────────────────────────────────────────────────────
GP_CONFIG = {
    "alpha": 1e-6,                 # noise regularisation
    "n_restarts_optimizer": 10,    # kernel optimisation restarts
    "normalize_y": True,           # handles negative/large-scale outputs
}

# ── ACQUISITION FUNCTION SCHEDULE ─────────────────────────────────────────────
AF_SCHEDULE = {
    "exploration":   {"weeks": range(1, 5),  "af": "ucb", "kappa": 2.576},
    "balanced":      {"weeks": range(5, 10), "af": "ei",  "kappa": None},
    "exploitation":  {"weeks": range(10,14), "af": "ucb", "kappa": 0.5},
}

print("✓ Configuration loaded")
print(f"  {len(FUNCTIONS)} functions registered")
for fn, cfg in FUNCTIONS.items():
    print(f"  F{fn}: {cfg['dim']}D — {cfg['description']}")


## 2. Core Functions — GP, Acquisition, Format

In [ ]:
# ── DATA LOADING ─────────────────────────────────────────────────────────────
def load_data(fn_id):
    """Load current (X, Y) data for a given function ID."""
    base = DATA_ROOT / f"function_{fn_id}"
    X = np.load(base / "initial_inputs.npy")
    Y = np.load(base / "initial_outputs.npy")
    return X, Y


def save_data(fn_id, X, Y):
    """Save updated (X, Y) back to .npy files."""
    base = DATA_ROOT / f"function_{fn_id}"
    base.mkdir(parents=True, exist_ok=True)
    np.save(base / "initial_inputs.npy", X)
    np.save(base / "initial_outputs.npy", Y)
    print(f"  ✓ F{fn_id} saved — {len(Y)} observations total")


# ── GP FITTING ────────────────────────────────────────────────────────────────
def fit_gp(X, Y):
    """
    Fit a Gaussian Process with Matern-5/2 kernel.
    Matern is more robust than RBF for real-world black-box functions.
    normalize_y=True handles negative and large-scale outputs correctly.
    """
    kernel = C(1.0, (1e-3, 1e3)) * Matern(
        length_scale=np.ones(X.shape[1]),
        length_scale_bounds=(1e-2, 10.0),
        nu=2.5
    )
    gp = GaussianProcessRegressor(
        kernel=kernel,
        alpha=GP_CONFIG["alpha"],
        n_restarts_optimizer=GP_CONFIG["n_restarts_optimizer"],
        normalize_y=GP_CONFIG["normalize_y"],
    )
    gp.fit(X, Y)
    return gp


# ── ACQUISITION FUNCTIONS ─────────────────────────────────────────────────────
def ucb_val(gp, x, kappa=2.576):
    """Upper Confidence Bound — maximisation: μ + κσ"""
    mu, sigma = gp.predict(x.reshape(1, -1), return_std=True)
    return mu[0] + kappa * sigma[0]


def ei_val(gp, x, f_best):
    """
    Expected Improvement — maximisation.
    Accounts for both probability AND magnitude of improvement.
    Always returns a non-negative value.
    """
    mu, sigma = gp.predict(x.reshape(1, -1), return_std=True)
    sigma = max(sigma[0], 1e-12)
    z = (mu[0] - f_best) / sigma
    improvement = (mu[0] - f_best) * norm.cdf(z) + sigma * norm.pdf(z)
    return max(improvement, 0.0)


def get_af_for_week(week):
    """Return acquisition function name and kappa based on current week."""
    for phase, cfg in AF_SCHEDULE.items():
        if week in cfg["weeks"]:
            return cfg["af"], cfg["kappa"], phase
    return "ei", None, "balanced"


# ── DIAGNOSTIC CHECKS ────────────────────────────────────────────────────────
def is_degenerate_kernel(gp, threshold=8.0):
    """
    True if any length_scale hit its upper bound.
    When length_scale → max, GP treats that dimension as flat (no signal detected).
    This typically causes the optimizer to drift to the boundary.
    """
    ls = np.atleast_1d(gp.kernel_.k2.length_scale)
    return bool(np.any(ls >= threshold))


def is_all_near_zero(Y, threshold=1e-8):
    """
    True when all observed outputs are essentially zero.
    Typically occurs in F1 (radiation) before the source is found.
    Standard GP/UCB is useless here — use pure uncertainty sampling instead.
    """
    return bool(np.abs(Y).max() < threshold)


# ── MAIN: FIND NEXT QUERY ─────────────────────────────────────────────────────
def find_next_query(gp, dim, f_best, week, Y, n_restarts=10):
    """
    Robust next-query finder with 4 safeguards:

    Safeguard 1 — All-near-zero Y (e.g. F1 before radiation source found):
        Switch to pure uncertainty maximisation. UCB/EI meaningless when f*≈0.

    Safeguard 2 — Degenerate kernel (length_scale hits upper bound):
        GP cannot detect signal in that dimension. Increase restarts and use
        Sobol initialisation for better coverage.

    Safeguard 3 — Strict boundary buffer (0.10 – 0.90):
        Prevents optimizer from collapsing to corners of the unit hypercube,
        which is a common artifact when the GP surface is nearly flat.

    Safeguard 4 — Post-optimisation boundary check:
        If any dimension is still within 1% of the buffer after optimisation,
        fall back to the maximum-uncertainty point as a safe exploration choice.
    """
    af_name, kappa, phase = get_af_for_week(week)
    BUF = 0.10   # 10% boundary buffer on each side
    bounds = [(BUF, 1.0 - BUF)] * dim

    degenerate = is_degenerate_kernel(gp)
    all_zero   = is_all_near_zero(Y)

    # ── Choose objective ──────────────────────────────────────────────
    if all_zero:
        # F1-type: no signal yet, explore maximally uncertain regions
        mode = "UNCERTAINTY [Y≈0 override]"
        def obj(x):
            _, s = gp.predict(x.reshape(1, -1), return_std=True)
            return -s[0]

    elif af_name == "ucb":
        mode = f"UCB κ={kappa}" + (" [degenerate kernel]" if degenerate else "")
        def obj(x):
            return -ucb_val(gp, x, kappa=kappa)

    else:  # ei
        mode = "EI" + (" [degenerate kernel]" if degenerate else "")
        def obj(x):
            return -ei_val(gp, x, f_best)

    # ── Run differential_evolution with multiple restarts ─────────────
    n = 15 if degenerate else n_restarts   # extra restarts for degenerate case

    best_result = None
    for seed in range(n):
        r = differential_evolution(
            obj, bounds,
            seed=seed * 7,
            maxiter=3000,
            popsize=25,
            tol=1e-14,
            mutation=(0.5, 1.5),
            recombination=0.9,
            init='sobol',   # better space coverage than random
            polish=True,    # final L-BFGS-B refinement
        )
        if best_result is None or r.fun < best_result.fun:
            best_result = r

    x_next = np.clip(best_result.x, 0.0, 0.999999)

    # ── Safeguard 4: boundary check ───────────────────────────────────
    at_boundary = np.any(
        (x_next < BUF + 0.01) | (x_next > 1.0 - BUF - 0.01)
    )

    if at_boundary and not all_zero:
        # Fall back to maximum uncertainty as a safe exploration choice
        def obj_unc(x):
            _, s = gp.predict(x.reshape(1, -1), return_std=True)
            return -s[0]
        r_fb = differential_evolution(
            obj_unc, bounds, seed=999,
            maxiter=2000, popsize=20,
            init='sobol', polish=True
        )
        x_fb = np.clip(r_fb.x, 0.0, 0.999999)
        fb_ok = not np.any((x_fb < BUF + 0.01) | (x_fb > 1.0 - BUF - 0.01))
        if fb_ok:
            x_next = x_fb
            mode += " → BOUNDARY_FALLBACK (max uncertainty)"

    mu_pred, sigma_pred = gp.predict(x_next.reshape(1, -1), return_std=True)
    ls = np.round(np.atleast_1d(gp.kernel_.k2.length_scale), 4)

    return {
        "x_next"       : x_next,
        "mu"           : mu_pred[0],
        "sigma"        : sigma_pred[0],
        "af_value"     : -best_result.fun,
        "af_name"      : af_name,
        "kappa"        : kappa,
        "phase"        : phase,
        "mode"         : mode,
        "length_scales": ls,
        "degenerate"   : degenerate,
        "all_zero"     : all_zero,
    }


# ── FORMAT FOR PORTAL ─────────────────────────────────────────────────────────
def format_submission(x):
    """Format x array as portal submission string: 0.123456-0.234567-..."""
    return "-".join([f"{v:.6f}" for v in x])


print("✓ All helper functions defined (v2 — with 4 safeguards)")
print()
print("  Safeguard 1: Y≈0 → pure uncertainty maximisation")
print("  Safeguard 2: Degenerate kernel → 15 restarts + Sobol init")
print("  Safeguard 3: Boundary buffer 0.10 – 0.90 (strict)")
print("  Safeguard 4: Post-check → fallback to max-uncertainty if still at boundary")


In [ ]:
# ── VISUALISATION HELPERS ────────────────────────────────────────────────────
def plot_function_summary(fn_id, X, Y, result=None, week=None):
    """
    Plot GP analysis for a single function.
    - 1D/2D: shows GP mean, uncertainty, observations, AF
    - Higher D: shows observation history + best found trend
    """
    dim = FUNCTIONS[fn_id]["dim"]
    desc = FUNCTIONS[fn_id]["description"]
    n_obs = len(Y)

    fig = plt.figure(figsize=(16, 5))
    fig.suptitle(
        f"Function {fn_id} — {desc}\n"
        f"Week {week or '?'} | {n_obs} observations | "
        f"Best y = {Y.max():.6f}",
        fontsize=12, fontweight='bold'
    )

    if dim == 2:
        # ── 2D heatmap of GP mean ──────────────────────────────────────
        gs = gridspec.GridSpec(1, 3, figure=fig, wspace=0.35)

        # GP Mean heatmap
        ax1 = fig.add_subplot(gs[0])
        res_grid = 60
        g = np.linspace(0, 1, res_grid)
        G1, G2 = np.meshgrid(g, g)
        grid_pts = np.column_stack([G1.ravel(), G2.ravel()])

        try:
            gp_temp = fit_gp(X, Y)
            mu_grid, sig_grid = gp_temp.predict(grid_pts, return_std=True)
            mu_map = mu_grid.reshape(res_grid, res_grid)
            sig_map = sig_grid.reshape(res_grid, res_grid)

            im1 = ax1.contourf(G1, G2, mu_map, levels=20, cmap='RdYlGn')
            plt.colorbar(im1, ax=ax1)
            ax1.scatter(X[:, 0], X[:, 1], c='white', s=40, zorder=5,
                       edgecolors='black', linewidths=0.8, label='Observations')
            best_idx = np.argmax(Y)
            ax1.scatter(X[best_idx, 0], X[best_idx, 1], c='gold', s=120,
                       marker='*', zorder=6, label=f'Best (y={Y.max():.4f})')
            if result:
                ax1.scatter(result['x_next'][0], result['x_next'][1],
                           c='cyan', s=120, marker='^', zorder=7,
                           label=f"Next ({result['af_name'].upper()})")
            ax1.set_xlabel('x₁'); ax1.set_ylabel('x₂')
            ax1.set_title('GP Posterior Mean μ(x)')
            ax1.legend(fontsize=7)

            # Uncertainty heatmap
            ax2 = fig.add_subplot(gs[1])
            im2 = ax2.contourf(G1, G2, sig_map, levels=20, cmap='Blues')
            plt.colorbar(im2, ax=ax2)
            ax2.scatter(X[:, 0], X[:, 1], c='white', s=40, zorder=5,
                       edgecolors='black', linewidths=0.8)
            if result:
                ax2.scatter(result['x_next'][0], result['x_next'][1],
                           c='red', s=120, marker='^', zorder=7)
            ax2.set_xlabel('x₁'); ax2.set_ylabel('x₂')
            ax2.set_title('GP Uncertainty σ(x)')
        except Exception as e:
            ax1.text(0.5, 0.5, f'GP fit error:\n{e}',
                    transform=ax1.transAxes, ha='center')
            ax2 = fig.add_subplot(gs[1])

        # Observation history
        ax3 = fig.add_subplot(gs[2])
        ax3.plot(range(1, n_obs+1), Y, 'o-', color='steelblue',
                markersize=5, linewidth=1.5, label='y per query')
        ax3.plot(range(1, n_obs+1),
                [max(Y[:i+1]) for i in range(n_obs)],
                's--', color='green', markersize=4, linewidth=1.5,
                label='Best so far')
        ax3.set_xlabel('Observation #'); ax3.set_ylabel('f(x)')
        ax3.set_title('Observation History')
        ax3.legend(fontsize=8)
        ax3.grid(True, alpha=0.3)

    else:
        # ── Higher D: observation history + parallel coordinates ──────
        gs = gridspec.GridSpec(1, 2, figure=fig, wspace=0.35)

        ax1 = fig.add_subplot(gs[0])
        ax1.plot(range(1, n_obs+1), Y, 'o-', color='steelblue',
                markersize=5, linewidth=1.5, label='y per query')
        ax1.plot(range(1, n_obs+1),
                [max(Y[:i+1]) for i in range(n_obs)],
                's--', color='green', markersize=4, linewidth=1.5,
                label='Best so far')
        ax1.set_xlabel('Observation #'); ax1.set_ylabel('f(x)')
        ax1.set_title('Observation History')
        ax1.legend(fontsize=8)
        ax1.grid(True, alpha=0.3)

        ax2 = fig.add_subplot(gs[1])
        # Top 5 best observations as bar chart of inputs
        top_k = min(5, n_obs)
        top_idx = np.argsort(Y)[-top_k:][::-1]
        x_top = X[top_idx]
        y_top = Y[top_idx]
        for rank, (xi, yi) in enumerate(zip(x_top, y_top)):
            ax2.bar(np.arange(dim) + rank * 0.15, xi, 0.12,
                   label=f'#{rank+1} y={yi:.4f}', alpha=0.8)
        ax2.set_xticks(np.arange(dim))
        ax2.set_xticklabels([f'x{i+1}' for i in range(dim)])
        ax2.set_ylabel('Input value (0-1)')
        ax2.set_title(f'Top-{top_k} Best Observations — Input Profiles')
        ax2.legend(fontsize=7)
        ax2.grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    plt.savefig(f"function_{fn_id}_week{week or 0}_analysis.png",
               dpi=150, bbox_inches='tight')
    plt.show()
    print(f"  Plot saved: function_{fn_id}_week{week or 0}_analysis.png")


print("✓ Visualisation functions defined")


## 3. Weekly Analysis — Run This Every Week

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# WEEKLY ANALYSIS — run once per week for all 8 functions
# ══════════════════════════════════════════════════════════════════════════════

CURRENT_WEEK = 1   # <── UPDATE THIS EACH WEEK (1 through 13)

print("=" * 70)
print(f"WEEK {CURRENT_WEEK} — BAYESIAN OPTIMISATION ANALYSIS")
print("=" * 70)

af_name, kappa, phase = get_af_for_week(CURRENT_WEEK)
print(f"Phase: {phase.upper()} | Acquisition: {af_name.upper()}", end="")
if kappa: print(f" | κ = {kappa}")
else: print()
print()

results = {}
submission_strings = {}

for fn_id in range(1, 9):
    cfg = FUNCTIONS[fn_id]
    dim = cfg["dim"]

    print(f"─" * 60)
    print(f"Function {fn_id} ({dim}D) — {cfg['description']}")
    print(f"Note: {cfg['notes']}")

    # ── Load data ──────────────────────────────────────────────────────
    try:
        X, Y = load_data(fn_id)
        n_init = cfg["n_init"]
        week_actual = len(Y) - n_init + 1  # infer week from data length
        print(f"  Loaded: {len(Y)} observations (initial={n_init}, added={len(Y)-n_init})")
        print(f"  Best y so far: {Y.max():.8f}  at x={X[np.argmax(Y)]}")
    except FileNotFoundError:
        print(f"  ✗ Data file not found at {DATA_ROOT}/function_{fn_id}/")
        print(f"    Create folder and place initial_inputs.npy + initial_outputs.npy")
        continue

    # ── Fit GP ────────────────────────────────────────────────────────
    print(f"  Fitting GP...", end=" ")
    gp = fit_gp(X, Y)
    print(f"done | kernel: {gp.kernel_}")

    # ── Find next query ───────────────────────────────────────────────
    print(f"  Optimising acquisition ({af_name.upper()})...", end=" ")
    result = find_next_query(gp, dim, Y.max(), CURRENT_WEEK, Y)
    print("done")

    mu_at_next, sig_at_next = gp.predict(
        result["x_next"].reshape(1, -1), return_std=True)

    print(f"  → Next x: {result['x_next']}")
    print(f"  → GP prediction: μ={result['mu']:.6f}  σ={result['sigma']:.6f}")
    print(f"  → Mode: {result['mode']}")
    print(f"  → Length scales: {result['length_scales']}")
    print(f"  → AF value: {result['af_value']:.6f}")

    # ── Portal format ──────────────────────────────────────────────────
    sub_str = format_submission(result["x_next"])
    submission_strings[fn_id] = sub_str
    print(f"  → PORTAL: {sub_str}")

    results[fn_id] = {"X": X, "Y": Y, "gp": gp, "result": result}
    print()

# ── Print all portal strings together ─────────────────────────────────────────
print("=" * 70)
print("COPY-PASTE INTO PORTAL")
print("=" * 70)
for fn_id in range(1, 9):
    if fn_id in submission_strings:
        dim = FUNCTIONS[fn_id]["dim"]
        print(f"  Function {fn_id} ({dim}-D):  {submission_strings[fn_id]}")
print("=" * 70)


## 4. Visualisation — GP Analysis Per Function

In [ ]:
# ── PLOT ALL FUNCTIONS ────────────────────────────────────────────────────────
# Run after the weekly analysis cell above

for fn_id, data in results.items():
    print(f"\nPlotting Function {fn_id}...")
    plot_function_summary(
        fn_id,
        data["X"], data["Y"],
        result=data["result"],
        week=CURRENT_WEEK
    )


## 5. Data Update — Run After Receiving Email Results

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# DATA UPDATE — run once after receiving the email with this week's outputs
# Fill in the y values you received from the portal email
# ══════════════════════════════════════════════════════════════════════════════

# ── FILL IN YOUR EMAIL RESULTS HERE ──────────────────────────────────────────
EMAIL_RESULTS = {
    # fn_id: y_value_from_email
    1: None,   # e.g. 0.00000123
    2: None,
    3: None,
    4: None,
    5: None,
    6: None,
    7: None,
    8: None,
}
# ─────────────────────────────────────────────────────────────────────────────

print(f"Updating data files for Week {CURRENT_WEEK}...")
print()

for fn_id, y_new in EMAIL_RESULTS.items():
    if y_new is None:
        print(f"  F{fn_id}: skipped (no value provided)")
        continue

    if fn_id not in results:
        print(f"  F{fn_id}: skipped (no analysis result found)")
        continue

    X_old = results[fn_id]["X"]
    Y_old = results[fn_id]["Y"]
    x_submitted = results[fn_id]["result"]["x_next"]

    # Append new observation
    X_new = np.vstack([X_old, x_submitted.reshape(1, -1)])
    Y_new = np.append(Y_old, y_new)

    # Save
    save_data(fn_id, X_new, Y_new)

    # Summary
    improved = y_new > Y_old.max()
    delta = y_new - Y_old.max()
    print(f"  F{fn_id}: y={y_new:.8f} | "
          f"{'✓ IMPROVEMENT' if improved else '✗ no improvement'} "
          f"| Δ={delta:+.6f} | best now={Y_new.max():.8f}")

print()
print("✓ Update complete. Run the weekly analysis cell again for next week.")


## 6. Deep Dive — Single Function Analysis

In [ ]:
# ── DEEP DIVE INTO ONE FUNCTION ──────────────────────────────────────────────
# Use this to investigate a specific function more carefully

FOCUS_FN = 1   # <── change function ID here

X, Y = load_data(FOCUS_FN)
cfg = FUNCTIONS[FOCUS_FN]
dim = cfg["dim"]

print(f"Deep dive: Function {FOCUS_FN} ({dim}D)")
print(f"Description: {cfg['description']}")
print(f"Notes: {cfg['notes']}")
print(f"Observations: {len(Y)}")
print()

# ── Summary statistics ────────────────────────────────────────────────────────
print("Y statistics:")
print(f"  min    = {Y.min():.8e}")
print(f"  max    = {Y.max():.8e}")
print(f"  mean   = {Y.mean():.8e}")
print(f"  std    = {Y.std():.8e}")
print(f"  n_neg  = {(Y < 0).sum()} / {len(Y)}")
print()

# ── All observations sorted by Y ──────────────────────────────────────────────
print("All observations (sorted by y, best first):")
sorted_idx = np.argsort(Y)[::-1]
for rank, idx in enumerate(sorted_idx):
    x_str = "  ".join([f"{v:.4f}" for v in X[idx]])
    print(f"  #{rank+1:2d}  y={Y[idx]:+.8e}  x=[{x_str}]")
print()

# ── GP fit and AF comparison ──────────────────────────────────────────────────
print("Fitting GP and comparing all acquisition functions...")
gp = fit_gp(X, Y)
f_best = Y.max()

results_af = {}
for af_name_test, kappa_test in [("UCB κ=2.58", 2.576), ("UCB κ=1.96", 1.96),
                                   ("UCB κ=0.50", 0.5),  ("EI", None)]:
    bounds = [(0.0, 0.999999)] * dim

    if kappa_test is not None:
        def neg_af_test(x, k=kappa_test):
            return -ucb(gp, x, kappa=k)
    else:
        def neg_af_test(x):
            return -ei(gp, x, f_best)

    res = differential_evolution(neg_af_test, bounds, seed=42,
                                  maxiter=1000, popsize=20, polish=True)
    x_opt = np.clip(res.x, 0.0, 0.999999)
    mu_v, sig_v = gp.predict(x_opt.reshape(1,-1), return_std=True)
    sub = format_submission(x_opt)
    results_af[af_name_test] = {"x": x_opt, "mu": mu_v[0], "sig": sig_v[0], "sub": sub}
    print(f"  {af_name_test:<14}  x={sub}  μ={mu_v[0]:+.6f}  σ={sig_v[0]:.6f}")

print()
print("Current week AF recommendation:")
_, _, phase = get_af_for_week(CURRENT_WEEK)
rec = find_next_query(gp, dim, f_best, CURRENT_WEEK)
print(f"  Phase: {rec['phase']} | AF: {rec['af_name'].upper()} | κ={rec['kappa']}")
print(f"  → {format_submission(rec['x_next'])}")


## 7. Progress Tracker — All Functions Over All Weeks

In [ ]:
# ── OVERALL PROGRESS SUMMARY ─────────────────────────────────────────────────
print("=" * 70)
print("OVERALL PROGRESS SUMMARY")
print("=" * 70)
print(f"{'Fn':<4} {'Dim':<5} {'Init':<6} {'Queries':<8} {'Best Y':<18} {'Best X (first 2 dims)'}")
print("-" * 70)

all_best = {}
for fn_id in range(1, 9):
    cfg = FUNCTIONS[fn_id]
    try:
        X, Y = load_data(fn_id)
        n_queries = len(Y) - cfg["n_init"]
        best_y = Y.max()
        best_x = X[np.argmax(Y)]
        x_str = "  ".join([f"{v:.4f}" for v in best_x[:2]])
        if cfg["dim"] > 2: x_str += " ..."
        print(f"  F{fn_id:<3} {cfg['dim']:<5} {cfg['n_init']:<6} {n_queries:<8} "
              f"{best_y:<18.8f} [{x_str}]")
        all_best[fn_id] = best_y
    except FileNotFoundError:
        print(f"  F{fn_id:<3} {cfg['dim']:<5} {'—':<6} {'—':<8} {'file not found'}")

print("-" * 70)
print()

# ── Bar chart of best y per function ─────────────────────────────────────────
if all_best:
    fig, ax = plt.subplots(figsize=(12, 4))
    fn_ids = list(all_best.keys())
    best_ys = list(all_best.values())
    colours = ['#2ecc71' if y > 0 else '#e74c3c' for y in best_ys]
    bars = ax.bar([f"F{i}" for i in fn_ids], best_ys, color=colours,
                  edgecolor='black', linewidth=0.8, alpha=0.85)
    for bar, y in zip(bars, best_ys):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
               f'{y:.4f}', ha='center', va='bottom', fontsize=9)
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_title(f'Best Observed Output Per Function — Week {CURRENT_WEEK}',
                fontsize=13, fontweight='bold')
    ax.set_ylabel('Best f(x) found')
    ax.set_xlabel('Function')
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig(f"progress_week{CURRENT_WEEK}.png", dpi=150, bbox_inches='tight')
    plt.show()
    print(f"  Progress chart saved: progress_week{CURRENT_WEEK}.png")


## 8. Weekly Reflection Template

In [ ]:
# ── WEEKLY REFLECTION ────────────────────────────────────────────────────────
# Fill this in after each submission and receiving results.
# This supports your GitHub documentation and final write-up.

reflection = {
    "week": CURRENT_WEEK,
    "af_used": af_name,
    "phase": phase,
    "rationale": """
    [Why did I choose this acquisition function this week?]
    [What did the GP uncertainty map tell me?]
    """,
    "observations": {
        # fn_id: "what did the new y value tell me?"
        1: "[e.g. Still near zero — source not found yet. Will try region X next week.]",
        2: "[e.g. Positive signal at 0.43, 0.67 — promising. GP pointed nearby.]",
    },
    "surprises": """
    [Anything unexpected in the results?]
    """,
    "next_week_strategy": """
    [Will I change AF? Change kappa? Focus on specific functions?]
    """,
    "best_results_so_far": {
        fn_id: {
            "best_y": all_best.get(fn_id, None),
        }
        for fn_id in range(1, 9)
    }
}

print(f"Week {reflection['week']} Reflection")
print(f"  AF used: {reflection['af_used'].upper()} | Phase: {reflection['phase']}")
print(f"  Rationale: {reflection['rationale'].strip()[:100]}...")
print()
print("  Best results so far:")
for fn_id, r in reflection["best_results_so_far"].items():
    if r["best_y"] is not None:
        print(f"    F{fn_id}: {r['best_y']:.8f}")
